# Cardiac MRI Image Segmentation

Walkthrough of the training pipeline: loading the ACDC dataset, training the segmentation
models (`UNet`, `CustomUNet`, `CustomSegNet`) and visualizing the results.

Run this notebook from the root of the repository.

## Setup

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(f"CUDA available: {torch.cuda.is_available()}")

## Dataset

The ACDC data is expected as an HDF5 file in the `data/` directory.

In [ ]:
from src.manage.HDF5Dataset import HDF5Dataset
import torchvision.transforms as transforms

# Set the HDF5 path according to your file location
acdc_hdf5_file = 'data/ift780_acdc.hdf5'

acdc_base_transform = transforms.Compose([
    transforms.ToTensor()
])
train_set = HDF5Dataset('train', acdc_hdf5_file, transform=acdc_base_transform)
test_set = HDF5Dataset('test', acdc_hdf5_file, transform=acdc_base_transform)

In [ ]:
print(f"Training slices: {len(train_set)}")
print(f"Test slices: {len(test_set)}")

In [ ]:
sample_input, sample_target = train_set[0]
print(f"Image shape: {tuple(sample_input.shape)}")
print(f"Mask shape: {tuple(sample_target.shape)}")

total_size_mb = (len(train_set) * sample_input.element_size() * sample_input.nelement()) / (1024 * 1024)
print(f"Approximate size of the training set in memory: {total_size_mb:.2f} MB")

## Training

All the options of the training script:

In [ ]:
!python train.py --help

### Reference UNet

In [ ]:
!python train.py --model UNet --num-epochs 2 --batch_size 32 --lr 0.01

### CustomUNet with Dice loss

In [ ]:
!python train.py --model CustomUNet --loss Dice --num-epochs 2 --batch_size 16 --lr 0.0001

### CustomSegNet with Focal loss

In [ ]:
!python train.py --model CustomSegNet --loss Focal --num-epochs 2 --batch_size 16 --lr 0.0001

## Prediction

Checkpoints are saved in `checkpoints/<model>_<loss>/`. The best model can be reloaded to
predict the masks of a few test images.

In [ ]:
!python train.py --model CustomUNet --loss Dice --predict checkpoints/CustomUNet_Dice/best_model_CustomUNet.pth

## Results

The training script saves the training curves and the predictions in `figures/`.

In [ ]:
from IPython.display import Image, display

image_names = [
    'figures/CustomUNet_metrics.png',
    'figures/CustomUNet_prediction_3.png',
    'figures/CustomUNet_prediction_4.png',
]

for image_name in image_names:
    display(Image(filename=image_name))